In [1]:
import os
while not os.path.isdir("DataTypes"):
    os.chdir("..")

import json
import multiprocessing

load("DataTypes/Grafo2Terminales.sage")
load("Scripts/CantidadesEstructurales.sage")
load("Scripts/Esqueleto.sage")

In [2]:
NUM_HILOS = 16
TAM_LOTE = 2000          # superadores por trabajo enviado a un proceso
ARCHIVO_SUPERADORES = "Datos/SuperadoresDelGlobo.json"

In [3]:
def EsCasiRegular(G):
    """ Grado máximo y grado mínimo difieren en 1 o menos. """
    grados = G.degree()
    return max(grados) - min(grados) <= 1

def EsMultipartitoCompleto(G):
    """
        G es multipartito completo sii la relación "no adyacentes o iguales"
        es de equivalencia, es decir, sii el complemento es unión disjunta
        de cliques (las partes). Incluye a los completos K_n (partes de tamaño 1).
    """
    return all(C.is_clique() for C in G.complement().connected_components_subgraphs())

def AnalizarLote(lote):
    """
        Trabajo de un proceso: recibe una lista de entradas {"s","t","t2","graph6"}
        y retorna (casi_regular, multipartito, {cantidad de puentes: cuántos},
        {partes del multipartito: cuántos}).
    """
    casi_regular, multipartito, puentes, tipos = 0, 0, {}, {}
    for e in lote:
        TC = Grafo2Terminales(Graph(e["graph6"]), e["s"], e["t"])
        esqueleto = ObtenerEsqueleto(TC).grafo
        casi_regular += EsCasiRegular(esqueleto)
        if EsMultipartitoCompleto(esqueleto):
            multipartito += 1
            partes = tuple(sorted((C.order() for C in esqueleto.complement().connected_components_subgraphs()), reverse=True))
            tipos[partes] = tipos.get(partes, 0) + 1
        k = int(b(TC))
        puentes[k] = puentes.get(k, 0) + 1
    return casi_regular, multipartito, puentes, tipos

def AnalizarSuperadores(superadores, num_hilos=NUM_HILOS):
    lotes = [superadores[i:i + TAM_LOTE] for i in range(0, len(superadores), TAM_LOTE)]
    with multiprocessing.get_context("fork").Pool(num_hilos) as pool:
        resultados = pool.map(AnalizarLote, lotes, chunksize=1)
    fila = {"X": len(superadores), "casi_regular": 0, "multipartito": 0, "puentes": {}, "tipos": {}}
    for cr, mp, puentes, tipos in resultados:
        fila["casi_regular"] += cr
        fila["multipartito"] += mp
        for k, v in puentes.items():
            fila["puentes"][k] = fila["puentes"].get(k, 0) + v
        for k, v in tipos.items():
            fila["tipos"][k] = fila["tipos"].get(k, 0) + v
    return fila

In [4]:
with open(ARCHIVO_SUPERADORES) as f:
    crudo = json.load(f)

columnas = ["(n, m)", "t2 globo", "X", "casi-regular", "multipartito", "0 puentes", "1 puente", "otros"]
anchos = [9, 10, 8, 12, 12, 9, 8, 7]
encabezado = " | ".join(c.rjust(w) for c, w in zip(columnas, anchos))
print(encabezado)
print("-" * len(encabezado))

tipos_multipartito = {}
for n in sorted(crudo, key=int):
    for m in sorted(crudo[n], key=int):
        d = crudo[n][m]
        f = AnalizarSuperadores(d["superadores"])
        p = f["puentes"]
        otros = sum(v for k, v in p.items() if k not in (0, 1))
        celdas = [str((int(n), int(m))), d["t2_globo"], f["X"], f["casi_regular"],
                  f["multipartito"], p.get(0, 0), p.get(1, 0), otros]
        print(" | ".join(str(c).rjust(w) for c, w in zip(celdas, anchos)))
        tipos_multipartito[(int(n), int(m))] = f["tipos"]

   (n, m) |   t2 globo |        X | casi-regular | multipartito | 0 puentes | 1 puente |   otros
------------------------------------------------------------------------------------------------
  (7, 11) |        200 |       61 |           14 |            0 |        26 |       35 |       0
  (7, 12) |        300 |      131 |           26 |            6 |        81 |       50 |       0
  (8, 16) |       2160 |     2196 |           46 |            5 |      1720 |      476 |       0
  (8, 17) |       3024 |     2585 |          101 |            5 |      2292 |      293 |       0
  (9, 22) |      28812 |    69330 |          239 |           16 |     66486 |     2844 |       0
  (9, 23) |      38416 |    58848 |          226 |           12 |     57589 |     1259 |       0
 (10, 29) |     458752 |  2432675 |         5551 |           13 |   2418973 |    13702 |       0
 (10, 30) |     589824 |  1711295 |          422 |           24 |   1706258 |     5037 |       0


In [5]:
# Detalle: qué multipartitos completos aparecen como esqueleto (partes -> cantidad de superadores)
for nm, tipos in tipos_multipartito.items():
    print(nm)
    for partes, cant in sorted(tipos.items(), key=lambda x: -x[1]):
        print(f"   K_{{{', '.join(map(str, partes))}}}: {cant}")

(7, 11)
(7, 12)
   K_{3, 2, 1}: 6
(8, 16)
   K_{3, 3, 1}: 4
   K_{4, 1, 1, 1}: 1
(8, 17)
   K_{3, 2, 2}: 5
(9, 22)
   K_{4, 2, 1, 1}: 9
   K_{3, 3, 2}: 7
(9, 23)
   K_{3, 3, 1, 1}: 7
   K_{4, 1, 1, 1, 1}: 4
   K_{5, 3, 1}: 1
(10, 29)
   K_{4, 2, 2, 1}: 12
   K_{5, 4, 1}: 1
(10, 30)
   K_{3, 3, 2, 1}: 12
   K_{4, 2, 1, 1, 1}: 12
